# Loading and Preprocessing:

In [120]:
import pandas as pd
G_train = pd.read_csv("GoEmotions/Processed/GoEmotions_train.csv")
G_test = pd.read_csv("GoEmotions/Processed/GoEmotions_test.csv")

S_train = pd.read_csv("SemEvalDataset/processed/train.csv")
S_test = pd.read_csv("SemEvalDataset/processed/test.csv")

T_train = pd.read_csv("TwitterEmotions/training.csv")
T_test = pd.read_csv("TwitterEmotions/test.csv")

In [121]:
G_train = G_train.rename(columns={'0': "text", '1': "label_id", '2': "example_id"})
G_test = G_test.rename(columns={'0': "text", '1': "label_id", '2': "example_id"})

In [122]:
emotions = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude",
    "grief", "joy", "love", "nervousness", "optimism", "pride",
    "realization", "relief", "remorse", "sadness", "surprise", "neutral"
]

def parse_labels(label_str):
    return [int(x) for x in str(label_str).split(",")]

def make_multilabel_columns(df):
    df = df.copy()

    # Convert "13,18" into [13, 18]
    df["label_list"] = df["label_id"].apply(parse_labels)

    # Add readable labels: [13,18] -> ["excitement", "love"]
    df["label_names"] = df["label_list"].apply(
        lambda ids: [emotions[i] for i in ids]
    )

    # Create one binary column per emotion
    for i, emotion in enumerate(emotions):
        df[emotion] = df["label_list"].apply(lambda labels: 1 if i in labels else 0)

    return df

G_train = make_multilabel_columns(G_train)
G_test = make_multilabel_columns(G_test)

# print(goemotions_train[["label_id", "label_list", "label_names"]].head())

emotion_cols = emotions
print(G_train.columns)
print(G_test.columns)

Index(['text', 'label_id', 'example_id', 'label_list', 'label_names',
       'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
       'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
       'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
       'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
       'relief', 'remorse', 'sadness', 'surprise', 'neutral'],
      dtype='str')
Index(['text', 'label_id', 'example_id', 'label_list', 'label_names',
       'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
       'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
       'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
       'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
       'relief', 'remorse', 'sadness', 'surprise', 'neutral'],
      dtype='str')


GoEmotions Statistics

In [58]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.preprocessing import StandardScaler

stop_words = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_text(text):
    return re.findall(r"\b[a-z]+\b", clean_text(text))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

def pad_tokens(tokens, max_len=100, padding="post"):
    tokens = tokens[:max_len]
    pad_amount = max_len - len(tokens)

    if padding == "post":
        return tokens + ["<PAD>"] * pad_amount
    return ["<PAD>"] * pad_amount + tokens

In [119]:
def preprocess_only(df, text_col, max_len=100):
    df = df.copy()

    df["clean_text"] = df[text_col].apply(clean_text)

    df["tokens"] = df[text_col].apply(tokenize_text)

    df["tokens_no_stopwords"] = df["tokens"].apply(remove_stopwords)

    df["padded_tokens"] = df["tokens_no_stopwords"].apply(
        lambda tokens: pad_tokens(tokens, max_len=max_len, padding="post")
    )

    df["word_count"] = df["tokens"].apply(len)
    df["clean_word_count"] = df["tokens_no_stopwords"].apply(len)
    df["char_count"] = df["clean_text"].apply(len)

    scaler = StandardScaler()
    df[["word_count_scaled", "clean_word_count_scaled", "char_count_scaled"]] = scaler.fit_transform(
        df[["word_count", "clean_word_count", "char_count"]]
    )

    return df

twitter_train = preprocess_only(T_train, text_col="text", max_len=100)
twitter_test = preprocess_only(T_test, text_col="text", max_len=100)

semeval_train = preprocess_only(S_train, text_col="Tweet", max_len=100)
semeval_test = preprocess_only(S_test, text_col="Tweet", max_len=100)

goemotions_train = preprocess_only(G_train, text_col="text", max_len=100)
goemotions_test = preprocess_only(G_test, text_col="text", max_len=100)

In [123]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer =TfidfVectorizer(
    max_features=30000,
    ngram_range=(1,2),
    min_df=2,
    sublinear_tf=True
)

X_train_text = vectorizer.fit_transform(
    goemotions_train["clean_text"]
)

X_test_text = vectorizer.transform(
    goemotions_test["clean_text"]
)

from sklearn.preprocessing import StandardScaler

metadata_cols = ["clean_word_count", "char_count"]

scaler = StandardScaler()

X_train_meta = scaler.fit_transform(goemotions_train[metadata_cols])
X_test_meta = scaler.transform(goemotions_test[metadata_cols])

from scipy.sparse import hstack, csr_matrix

X_train = hstack([X_train_text, csr_matrix(X_train_meta)])
X_test = hstack([X_test_text, csr_matrix(X_test_meta)])

Y_train = goemotions_train[emotions]
Y_test = goemotions_test[emotions]

# Model Training

## Simpler Models

In [124]:
import time
import pandas as pd

from sklearn.svm import LinearSVC

from sklearn.multioutput import MultiOutputClassifier

from sklearn.metrics import classification_report, f1_score, accuracy_score

Model Training Helper Method

In [129]:
def train_and_evaluate_model(model, model_name, X_train, Y_train, X_test, Y_test):
    start_train = time.time()
    model.fit(X_train, Y_train)
    train_time = time.time() - start_train

    start_pred = time.time()
    Y_pred = model.predict(X_test)
    inference_time = time.time() - start_pred

    micro_f1 = f1_score(Y_test, Y_pred, average="micro", zero_division=0)
    macro_f1 = f1_score(Y_test, Y_pred, average="macro", zero_division=0)
    accuracy = accuracy_score(Y_test, Y_pred)

    print(f"\n===== {model_name} =====")
    print(f"Training Time: {train_time:.2f} seconds")
    print(f"Inference Time: {inference_time:.2f} seconds")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Micro F1: {micro_f1:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(
        Y_test,
        Y_pred,
        target_names=emotions,
        zero_division=0
    ))

    return {
        "model_name": model_name,
        "model": model,
        "train_time": train_time,
        "inference_time": inference_time,
        "accuracy": accuracy,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "Y_pred": Y_pred
    }

Naive Bayes

In [130]:
from sklearn.model_selection import GridSearchCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import MultinomialNB

nb_model = OneVsRestClassifier(
    MultinomialNB(
        alpha=0.1
    )
)

nb_results = train_and_evaluate_model(
    nb_model,
    "Naive Bayes",
    X_train_text,
    Y_train,
    X_test_text,
    Y_test
)


===== Naive Bayes =====
Training Time: 0.44 seconds
Inference Time: 0.05 seconds
Accuracy: 0.1585
Micro F1: 0.2590
Macro F1: 0.1118

Classification Report:
                precision    recall  f1-score   support

    admiration       0.81      0.21      0.33       504
     amusement       0.89      0.09      0.16       264
         anger       0.71      0.06      0.11       198
     annoyance       0.33      0.00      0.01       320
      approval       0.67      0.03      0.07       351
        caring       0.88      0.05      0.10       135
     confusion       0.40      0.01      0.03       153
     curiosity       0.47      0.03      0.06       284
        desire       1.00      0.01      0.02        83
disappointment       0.00      0.00      0.00       151
   disapproval       0.83      0.02      0.04       267
       disgust       1.00      0.07      0.12       123
 embarrassment       0.00      0.00      0.00        37
    excitement       0.33      0.03      0.05       103
  

SVM Model

In [131]:
svm_model = OneVsRestClassifier(
    LinearSVC(
        C=1.0,
        class_weight="balanced",
        max_iter=5000
    )
)

svm_results = train_and_evaluate_model(
    svm_model,
    "Linear SVM",
    X_train,
    Y_train,
    X_test,
    Y_test
)


===== Linear SVM =====
Training Time: 12.34 seconds
Inference Time: 0.01 seconds
Accuracy: 0.2908
Micro F1: 0.4994
Macro F1: 0.4052

Classification Report:
                precision    recall  f1-score   support

    admiration       0.53      0.64      0.58       504
     amusement       0.69      0.80      0.74       264
         anger       0.37      0.41      0.39       198
     annoyance       0.22      0.32      0.26       320
      approval       0.20      0.31      0.24       351
        caring       0.22      0.26      0.24       135
     confusion       0.27      0.39      0.32       153
     curiosity       0.34      0.46      0.39       284
        desire       0.46      0.36      0.41        83
disappointment       0.20      0.21      0.20       151
   disapproval       0.25      0.36      0.29       267
       disgust       0.45      0.41      0.43       123
 embarrassment       0.26      0.19      0.22        37
    excitement       0.21      0.32      0.26       103
  

Random Forest

In [132]:
from sklearn.ensemble import RandomForestClassifier

rf_model = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
)

rf_results = train_and_evaluate_model(
    rf_model,
    "Random Forest",
    X_train,
    Y_train,
    X_test,
    Y_test
)


===== Random Forest =====
Training Time: 216.51 seconds
Inference Time: 6.70 seconds
Accuracy: 0.2692
Micro F1: 0.4993
Macro F1: 0.4014

Classification Report:
                precision    recall  f1-score   support

    admiration       0.46      0.70      0.55       504
     amusement       0.71      0.87      0.78       264
         anger       0.36      0.51      0.42       198
     annoyance       0.25      0.34      0.29       320
      approval       0.26      0.39      0.31       351
        caring       0.20      0.45      0.28       135
     confusion       0.24      0.55      0.33       153
     curiosity       0.31      0.59      0.41       284
        desire       0.26      0.45      0.33        83
disappointment       0.19      0.21      0.20       151
   disapproval       0.23      0.48      0.31       267
       disgust       0.42      0.50      0.46       123
 embarrassment       0.22      0.11      0.15        37
    excitement       0.20      0.45      0.27       10

In [133]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

lr_model = OneVsRestClassifier(
    LogisticRegression(
        C=10,
        class_weight="balanced",
        max_iter=3000,
        solver="liblinear"
    )
)

lr_results = train_and_evaluate_model(
    lr_model,
    "Logistic Regression",
    X_train,
    Y_train,
    X_test,
    Y_test
)



===== Logistic Regression =====
Training Time: 8.69 seconds
Inference Time: 0.01 seconds
Accuracy: 0.2965
Micro F1: 0.5082
Macro F1: 0.4286

Classification Report:
                precision    recall  f1-score   support

    admiration       0.55      0.66      0.60       504
     amusement       0.70      0.83      0.76       264
         anger       0.36      0.45      0.40       198
     annoyance       0.24      0.33      0.27       320
      approval       0.20      0.32      0.25       351
        caring       0.24      0.33      0.28       135
     confusion       0.28      0.46      0.35       153
     curiosity       0.34      0.48      0.40       284
        desire       0.43      0.42      0.42        83
disappointment       0.20      0.25      0.22       151
   disapproval       0.26      0.40      0.31       267
       disgust       0.43      0.46      0.44       123
 embarrassment       0.27      0.24      0.26        37
    excitement       0.21      0.36      0.27     

Comparison of results

In [134]:
results_df = pd.DataFrame([
    {
        "Model": nb_results["model_name"],
        "Training Time": nb_results["train_time"],
        "Inference Time": nb_results["inference_time"],
        "Accuracy": nb_results["accuracy"],
        "Micro F1": nb_results["micro_f1"],
        "Macro F1": nb_results["macro_f1"]
    },
    {
        "Model": svm_results["model_name"],
        "Training Time": svm_results["train_time"],
        "Inference Time": svm_results["inference_time"],
        "Accuracy": svm_results["accuracy"],
        "Micro F1": svm_results["micro_f1"],
        "Macro F1": svm_results["macro_f1"]
    },
    {
        "Model": rf_results["model_name"],
        "Training Time": rf_results["train_time"],
        "Inference Time": rf_results["inference_time"],
        "Accuracy": rf_results["accuracy"],
        "Micro F1": rf_results["micro_f1"],
        "Macro F1": rf_results["macro_f1"]
    },
    {
        "Model": lr_results["model_name"],
        "Training Time": lr_results["train_time"],
        "Inference Time": lr_results["inference_time"],
        "Accuracy": lr_results["accuracy"],
        "Micro F1": lr_results["micro_f1"],
        "Macro F1": lr_results["macro_f1"]
    }
])

print(results_df)

                 Model  Training Time  Inference Time  Accuracy  Micro F1  \
0          Naive Bayes       0.441265        0.049848  0.158467  0.258985   
1           Linear SVM      12.343631        0.014063  0.290768  0.499399   
2        Random Forest     216.513114        6.697874  0.269210  0.499302   
3  Logistic Regression       8.686665        0.009707  0.296481  0.508202   

   Macro F1  
0  0.111771  
1  0.405185  
2  0.401436  
3  0.428617  


Per-Emotion F1

In [ ]:
from sklearn.metrics import f1_score

for i, emotion in enumerate(emotions):
    score = f1_score(
        Y_test.iloc[:, i],
        svm_results["Y_pred"][:, i]
    )
    print(emotion, score)

## Train Complex Models

Prepare Data

In [22]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score

In [28]:
model_name = "bert-base-uncased"
# Later use: "roberta-base"

train_df = goemotions_train[["clean_text"] + emotions].copy()
test_df = goemotions_test[["clean_text"] + emotions].copy()

train_df["labels"] = train_df[emotions].astype(float).values.tolist()
test_df["labels"] = test_df[emotions].astype(float).values.tolist()

train_dataset = Dataset.from_pandas(train_df[["clean_text", "labels"]])
test_dataset = Dataset.from_pandas(test_df[["clean_text", "labels"]])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.remove_columns(["clean_text"])
test_dataset = test_dataset.remove_columns(["clean_text"])

train_dataset.set_format("torch")
test_dataset.set_format("torch")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(emotions),
    problem_type="multi_label_classification"
)

In [31]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    return {
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "subset_accuracy": accuracy_score(labels, preds)
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert_goemotions_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    logging_dir="./logs",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
bert_results = trainer.evaluate()
print(bert_results)